In [1]:
import os
import hashlib
import pickle
import numpy as np
import requests
from dataclasses import dataclass, field
from typing import List, Dict, Any

from sklearn.neighbors import NearestNeighbors

import pyAutoSummarizer.base as psr
from tqdm import tqdm


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
TRAIN_PATH = "train-2.tab"
TEST_PATH = "test.tab"

DOCUMENT_COL = None   # es. "text" -- lascia None per auto-detect
SUMMARY_COL = None    # es. "highlights" -- lascia None per auto-detect
ID_COL = None
SEP = "\t"  # sia .tab che .tsv sono di norma tab-separated


In [ ]:
import csv as _csv
import sys as _sys
import pandas as pd

_csv.field_size_limit(_sys.maxsize)


def load_tabular_dataset(path, sep="\t", document_col=None, summary_col=None, id_col=None):
    df = pd.read_csv(path, sep=sep, engine="python", dtype=str, keep_default_na=False)
    print(f"[{path}] colonne trovate: {list(df.columns)}  (righe: {len(df)})")

    def _guess(candidates, cols):
        for c in candidates:
            if c in cols:
                return c
        return None

    cols = list(df.columns)
    doc_col = document_col or _guess(
        ["document", "documento", "text", "source", "src", "article"], cols)
    sum_col = summary_col or _guess(
        ["summary", "riassunto", "target", "reference", "highlights", "abstract"], cols)
    _id_col = id_col or _guess(["id", "ID", "index"], cols)

    if doc_col is None or sum_col is None:
        raise ValueError(
            f"Non riesco a capire quali colonne siano document/summary in {path}. "
            f"Colonne disponibili: {cols}. Specifica document_col e summary_col esplicitamente."
        )

    records = []
    for i, row in df.iterrows():
        records.append({
            "id": row[_id_col] if _id_col else i,
            "document": row[doc_col],
            "summary": row[sum_col],
        })
    return records


train_set = load_tabular_dataset(TRAIN_PATH, sep=SEP, document_col=DOCUMENT_COL,
                                  summary_col=SUMMARY_COL, id_col=ID_COL)
test_set_small = load_tabular_dataset(TEST_PATH, sep=SEP, document_col=DOCUMENT_COL,
                                       summary_col=SUMMARY_COL, id_col=ID_COL)

print(f"Training set: {len(train_set)} documenti")
print(f"Test set ridotto: {len(test_set_small)} documenti")

[train-2.tab] colonne trovate: ['document', 'summary']  (righe: 44882)
[test.tab] colonne trovate: ['document', 'summary']  (righe: 5612)
Training set: 44882 documenti
Test set ridotto: 5612 documenti


In [6]:
# --- Configurazione few-shot --------------------------------------------------
import os

import summ_utils as su

METODO_FS  = 'qwen_fewshot'
SCOPE_FS   = os.environ.get('SUMM_SCOPE', 'test')
SEED_FS    = 42
# es. 3 per uno smoke test rapido; None = tutti; SUMM_LIMIT usato da run_benchmark_test.py
LIMIT_FS   = int(os.environ['SUMM_LIMIT']) if 'SUMM_LIMIT' in os.environ else None

MODELLO_FS    = 'qwen2.5:7b-instruct'      # tag ollama; verificare con `ollama list`
OLLAMA_URL_FS = 'http://localhost:11434/v1'  # endpoint OpenAI-compatibile di ollama

MAX_TOKENS_FS  = 200
TEMPERATURE_FS = 0.3

# --- Tecnica di embedding e numero di esempi few-shot ---
EMBEDDING_TECNICA_FS = 'sbert_mpnet_en'   # all-mpnet-base-v2, via build_embedding_techniques()
K_FS = 4

# --- Prompt: variante "new" (vedi build_fewshot_prompt_new) ---
PROMPT_VARIANTE_FS = 'new'
ETICHETTA_FS   = 'Qwen few-shot '
NOTE_CONFIG_FS = (
    f'few-shot k={K_FS}, embedding={EMBEDDING_TECNICA_FS}, prompt={PROMPT_VARIANTE_FS}; '
    f'nearest neighbours dal training set (train-2.tab), test set completo (test.tab), '
    f'max_tokens={MAX_TOKENS_FS}'
)

# --- Percorsi output, ancora via summ_utils per coerenza col resto del progetto ---
BASE_FS = su.trova_base_dir()
P_FS    = su.percorsi_standard(BASE_FS)
OUT_PATH_FS = P_FS['summaries_dir'] / f'{METODO_FS}_{SCOPE_FS}.tsv'

# --- Test set completo, caricato direttamente da test.tab ---
TEST_FULL_PATH_FS = "test.tab"
test_set_fs = load_tabular_dataset(
    TEST_FULL_PATH_FS, sep=SEP, document_col=DOCUMENT_COL,
    summary_col=SUMMARY_COL, id_col=ID_COL,
)
test_set_fs = [{**d, 'row_id': d['id']} for d in test_set_fs]
if LIMIT_FS is not None:
    test_set_fs = test_set_fs[:LIMIT_FS]

print(f'Modello   : {MODELLO_FS} via {OLLAMA_URL_FS}')
print(f'Ambito    : {SCOPE_FS} ({len(test_set_fs)} documenti da {TEST_FULL_PATH_FS})')
print(f'Embedding : {EMBEDDING_TECNICA_FS} | k={K_FS} | prompt={PROMPT_VARIANTE_FS}')
print(f'Output    : {OUT_PATH_FS}')

[test.tab] colonne trovate: ['document', 'summary']  (righe: 5612)
Modello   : qwen2.5:7b-instruct via http://localhost:11434/v1
Ambito    : test (5612 documenti da test.tab)
Embedding : sbert_mpnet_en | k=4 | prompt=new
Output    : /Users/fed/Desktop/Master_AI4STEM/results/summaries/qwen_fewshot_test.tsv


In [ ]:
import hashlib
from pathlib import Path

from sentence_transformers import SentenceTransformer
import numpy as np

NOME_MODELLO_EMBEDDING = "all-mpnet-base-v2"
CACHE_DIR_EMBEDDING = P_FS['sample_dir'].parent / 'embeddings_cache'  # adjust if you prefer another folder
CACHE_DIR_EMBEDDING.mkdir(parents=True, exist_ok=True)

_modello_embedding = None

def get_modello_embedding():
    global _modello_embedding
    if _modello_embedding is None:
        _modello_embedding = SentenceTransformer(NOME_MODELLO_EMBEDDING)
    return _modello_embedding


def _chiave_cache(testi, nome_modello):
    """Hash of texts + model name: any change in either invalidates the cache."""
    h = hashlib.sha256()
    h.update(nome_modello.encode('utf-8'))
    for t in testi:
        h.update(b'\x00')  # separator, avoids accidental collisions between texts
        h.update(t.encode('utf-8'))
    return h.hexdigest()[:16]


def calcola_embeddings_cached(testi, etichetta, batch_size=32):
    """etichetta: short label for the filename (e.g. 'train', 'test_fs') —
    just for a readable cache filename, not part of the cache-validity check."""
    chiave = _chiave_cache(testi, NOME_MODELLO_EMBEDDING)
    cache_path = CACHE_DIR_EMBEDDING / f'{etichetta}_{NOME_MODELLO_EMBEDDING}_{chiave}.npy'

    if cache_path.exists():
        print(f'[{etichetta}] embeddings from cache ({cache_path.name})')
        return np.load(cache_path)

    print(f'[{etichetta}] computing embeddings (not cached yet)...')
    modello = get_modello_embedding()
    emb = modello.encode(
        testi, batch_size=batch_size, show_progress_bar=True,
        convert_to_numpy=True, normalize_embeddings=True,
    )
    np.save(cache_path, emb)
    return emb



In [10]:
# =========================================================================
# 2. NEAREST NEIGHBOURS
# =========================================================================

def get_nearest_neighbours(train_embeddings: np.ndarray, test_embeddings: np.ndarray, k: int) -> np.ndarray:
    nn = NearestNeighbors(n_neighbors=k, metric="cosine")
    nn.fit(train_embeddings)
    _, indices = nn.kneighbors(test_embeddings)
    return indices

In [ ]:
# =========================================================================
# 3.PROMPT
# =========================================================================

def build_fewshot_prompt_new(neighbour_examples, target_document):
    n = len(neighbour_examples)
    blocks = []
    for i, ex in enumerate(neighbour_examples, 1):
        blocks.append(f"--- Example {i} ---\nArticle:\n{ex['document']}\n\nSummary:\n{ex['summary']}")
    examples_text = "\n\n".join(blocks)

    return (
        "You are a professional news editor who writes concise, accurate summaries.\n\n"
        f"Below are {n} example pairs of (article -> human-written summary). "
        "Study their length, tone, and level of detail closely — your summary must match this style.\n\n"
        f"{examples_text}\n\n"
        "=== New article to summarize ===\n"
        f"{target_document}\n\n"
        "Instructions:\n"
        f"- Write a summary of ONLY the article above, matching the style and approximate length of the {n} example summaries.\n"
        "- Do not include any facts, names, or details from the example articles — only from the new article.\n"
        "- Do not add a title, preamble, or explanation — output ONLY the summary text itself.\n\n"
        "Summary:"
    )

In [12]:
# --- Budget a lunghezza del riferimento (issue #16), versione few-shot -------
# Stessa logica dello zero-shot: con SUMM_SCOPE='test_budgetref' il budget in
# parole del cluster (= lunghezza del riferimento) entra nel prompt e il limite
# in token sale a MAX_TOKENS_BUDGET_FS; None = comportamento storico (nessun
# aggiustamento). Il prompt a budget e' DERIVATO da build_fewshot_prompt_new
# (stesse istruzioni + la sola lunghezza), non riscritto a mano.
BUDGET_FS = su.budget_riferimento if su.budget_attivo(SCOPE_FS) else None

# Stesso fattore calibrato sul pilota di 30 righe con qwen (issue #16): con
# "approximately N words" ~0,65 N (13% in banda); con "about N ... at least N"
# ~0,83 N (53%); chiedendo 1,2 N con lo stesso vincolo la mediana arriva a
# 1,05 N e, dopo il tetto a 1,25 N, l'87% delle righe e' in banda. Il fattore
# e' una calibrazione, non un principio: va dichiarato, e ripilotato per ogni modello.
FATTORE_RICHIESTA_FS = 1.2
MAX_TOKENS_BUDGET_FS = 1500   # a budget: un bersaglio da 300 parole non ci sta in 400 token


def build_fewshot_prompt_new_budget(neighbour_examples, target_document, richiesto):
    """Variante a budget di build_fewshot_prompt_new: stesse istruzioni,
    con l'aggiunta del vincolo di lunghezza in parole."""
    n = len(neighbour_examples)
    blocks = []
    for i, ex in enumerate(neighbour_examples, 1):
        blocks.append(f"--- Example {i} ---\nArticle:\n{ex['document']}\n\nSummary:\n{ex['summary']}")
    examples_text = "\n\n".join(blocks)

    return (
        "You are a professional news editor who writes concise, accurate summaries.\n\n"
        f"Below are {n} example pairs of (article -> human-written summary). "
        "Study their length, tone, and level of detail closely — your summary must match this style.\n\n"
        f"{examples_text}\n\n"
        "=== New article to summarize ===\n"
        f"{target_document}\n\n"
        "Instructions:\n"
        f"- Write a summary of ONLY the article above, of about {richiesto} words. "
        f"The summary must be at least {richiesto} words long.\n"
        "- Do not include any facts, names, or details from the example articles — only from the new article.\n"
        "- Do not add a title, preamble, or explanation — output ONLY the summary text itself.\n\n"
        "Summary:"
    )


# Verifica che la derivazione sia effettivamente diversa dal prompt base (stesso
# controllo di sicurezza che avevi nello zero-shot, adattato al few-shot)
_probe_examples = [{"document": "x", "summary": "y"}]
assert build_fewshot_prompt_new_budget(_probe_examples, "z", 42) != build_fewshot_prompt_new(_probe_examples, "z"), \
    'build_fewshot_prompt_new cambiato: aggiornare la derivazione budget'

print(f'Budget FS : {"lunghezza del riferimento nel prompt, max_tokens=" + str(MAX_TOKENS_BUDGET_FS) if BUDGET_FS else "nessuno, max_tokens=" + str(MAX_TOKENS_FS)}')

Budget FS : nessuno, max_tokens=200


In [ ]:
from openai import OpenAI

client_fs = OpenAI(base_url=OLLAMA_URL_FS, api_key='ollama')  # la chiave e' ignorata da ollama


# --- Precalcolo dei nearest neighbours per ogni documento del test set ------
# (necessario perché genera_fs riceve solo il testo, non l'indice/posizione,
# e ciclo_summarization può eseguire le chiamate in parallelo via ThreadPoolExecutor)


train_docs = [su.prepara_documento(d["document"]) for d in train_set]
train_summaries = [d["summary"] for d in train_set]
test_docs_fs = [su.prepara_documento(d["document"]) for d in test_set_fs]

train_emb_fs = calcola_embeddings_cached(train_docs, 'train')
test_emb_fs = calcola_embeddings_cached(test_docs_fs, 'test_fs')
neighbour_idx_fs = get_nearest_neighbours(train_emb_fs, test_emb_fs, K_FS)

NEIGHBOUR_MAP_FS = {
    doc: [
        {"document": train_docs[j], "summary": train_summaries[j]}
        for j in neighbour_idx_fs[i]
    ]
    for i, doc in enumerate(test_docs_fs)
}


def genera_fs(documento, budget=None):
    examples = NEIGHBOUR_MAP_FS.get(documento)
    if examples is None:
        raise RuntimeError("Documento non trovato tra i nearest neighbours precalcolati "
                            "(possibile disallineamento tra test_set_fs e la mappa)")

    if budget is None:
        prompt = build_fewshot_prompt_new(examples, documento)
        max_tok = MAX_TOKENS_FS
    else:
        richiesto = int(round(FATTORE_RICHIESTA_FS * budget))
        prompt = build_fewshot_prompt_new_budget(examples, documento, richiesto)
        max_tok = MAX_TOKENS_BUDGET_FS

    risposta = client_fs.chat.completions.create(
        model=MODELLO_FS,
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=max_tok,
        temperature=TEMPERATURE_FS)
    scelta = risposta.choices[0]
    contenuto = scelta.message.content
    if not contenuto or not contenuto.strip():
        # solleva -> il ciclo condiviso registra l'errore e NON scrive la riga
        raise RuntimeError(f'risposta vuota (finish_reason={scelta.finish_reason})')
    return contenuto.strip()


scrittore_fs = su.ScrittoreRiassunti(OUT_PATH_FS)
errori_fs = su.ciclo_summarization(test_set_fs, scrittore_fs, genera_fs, limit=LIMIT_FS,
                                    etichetta=ETICHETTA_FS, budget=BUDGET_FS)
scrittore_fs.chiudi()

[train] embeddings from cache (train_all-mpnet-base-v2_69eb11febad51365.npy)
[test_fs] embeddings from cache (test_fs_all-mpnet-base-v2_fa70ddcaa8a460a8.npy)
Qwen few-shot [1] media 18.4 s/esempio (saltati 161 gia' fatti)
Qwen few-shot [2] media 18.1 s/esempio (saltati 161 gia' fatti)
Qwen few-shot [3] media 18.5 s/esempio (saltati 161 gia' fatti)
Qwen few-shot [10] media 21.1 s/esempio (saltati 161 gia' fatti)
Qwen few-shot [20] media 22.7 s/esempio (saltati 161 gia' fatti)
Qwen few-shot [30] media 23.6 s/esempio (saltati 161 gia' fatti)
Qwen few-shot [40] media 24.8 s/esempio (saltati 161 gia' fatti)
Qwen few-shot [50] media 25.2 s/esempio (saltati 161 gia' fatti)
Qwen few-shot [60] media 25.1 s/esempio (saltati 161 gia' fatti)
Qwen few-shot [70] media 24.8 s/esempio (saltati 161 gia' fatti)
Qwen few-shot [80] media 24.4 s/esempio (saltati 161 gia' fatti)
Qwen few-shot [90] media 24.2 s/esempio (saltati 161 gia' fatti)
Qwen few-shot [100] media 24.2 s/esempio (saltati 161 gia' fatti)

In [38]:
import json
from bert_score import score as bert_score_fn

import pandas as pd

riferimenti_fs = [
    {"row_id": i, "summary": es["summary"], "split": "test"}
    for i, es in enumerate(test_set_fs)
]

# --- carica quanto appena generato da genera_fs -----------------------------
riassunti_fs   = su.carica_riassunti(OUT_PATH_FS)
#riferimenti_fs = esempi_scope()

config_fs = {'modello': MODELLO_FS,
             'backend': 'ollama (endpoint OpenAI-compatibile)',
             'max_tokens': MAX_TOKENS_BUDGET_FS if BUDGET_FS else MAX_TOKENS_FS,
             'temperature': TEMPERATURE_FS,
             'note': NOTE_CONFIG_FS}

# --- ROUGE / BLEU / METEOR con la pipeline già usata per il resto del progetto
righe_fs, aggregato_fs = su.valuta_e_salva(riferimenti_fs, riassunti_fs, METODO_FS, SCOPE_FS,
                                            P_FS['metrics_dir'], config_fs)
# --- BERTScore (non calcolato da su.valuta_e_salva) --------------------------
row_id_to_ref = {r["row_id"]: r["summary"] for r in riferimenti_fs}

refs = [row_id_to_ref[r["row_id"]] for r in righe_fs]
hyps = [riassunti_fs[r["row_id"]] for r in righe_fs]

P, R, F = bert_score_fn(hyps, refs, model_type='roberta-large', lang='en',
                        batch_size=16, verbose=True)

metriche_finali = {
    'rouge1_f1':   aggregato_fs['overall']['rouge1_f1'],
    'rouge2_f1':   aggregato_fs['overall']['rouge2_f1'],
    'meteor':      aggregato_fs['overall']['meteor'],
    'bleu':        aggregato_fs['overall']['bleu'],
    'bertscore_f1': float(F.mean()),
}
print(json.dumps(metriche_finali, indent=2))

Metriche per-esempio : /Users/fed/Desktop/Master_AI4STEM/results/metrics/qwen_fewshot_test_per_example.csv (5612 righe)
Metriche aggregate   : /Users/fed/Desktop/Master_AI4STEM/results/metrics/qwen_fewshot_test_aggregate.json


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 25685.34it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


100%|██████████| 702/702 [19:17<00:00,  1.65s/it]


computing greedy matching.


100%|██████████| 351/351 [00:04<00:00, 71.19it/s]


done in 1162.55 seconds, 4.83 sentences/sec
{
  "rouge1_f1": 0.3214587719087689,
  "rouge2_f1": 0.09967457050862762,
  "meteor": 0.274275133918256,
  "bleu": 0.04636508750219048,
  "bertscore_f1": 0.8620166182518005
}
